In [2]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("../data/cc_model.csv", parse_dates=["date_received"])

print(df.shape)
print(df["split"].value_counts())
df.head(3)

(81996, 6)
split
train    45130
test     22208
valid    14658
Name: count, dtype: int64


,complaint_id,date_received,narrative,issue,label,split
0,5931539,2022-08-30,I was booking a flight on the XXXX website on ...,"Advertising and marketing, including promotion...","Advertising and marketing, including promotion...",test
1,5887572,2022-08-17,"In late XX/XX/2022, my friend referred me for ...","Advertising and marketing, including promotion...","Advertising and marketing, including promotion...",test
2,6145363,2022-10-29,I did a balance transfer and paid a fee of XXX...,"Advertising and marketing, including promotion...","Advertising and marketing, including promotion...",test


In [5]:
for i in range(3):
    print(f"--- {df.loc[i, 'label']} ---")
    print(df.loc[i, "narrative"][:600], "\n")

--- Advertising and marketing, including promotional offers ---
I was booking a flight on the XXXX website on XXXX XXXX XXXX for a flight trip on XXXX XXXX It showed me a promotional where it showed if I were to apply for the " DELTA SKYMILES GOLD AMERICAN EXPRESS CARD '' and get approved I would XXXX $ statement credit fir this flight booking and would be able to use the card this booking. But aftwr I got approved I was not given any instant account information and after spending 2-3 hours with various agents on calls who provided fake promises saying I could get an expedited delivery only to be negated by the next agent. 
I would not apply for this card  

--- Advertising and marketing, including promotional offers ---
In late XX/XX/2022, my friend referred me for American Express XXXX XXXX XXXX credit card and I was offered with a welcome bonus of XXXX XXXX ( my friend witnessed it ) if I can spend {$3000.00} in 3 months. However I still have not received the welcome bonus after I h

In [9]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\{\$[\d,\.]+\}", " dollar_amount ", text)   # {$450.00} -> one token
    text = re.sub(r"X{2,}(/X{2,})*", " ", text)                  # remove XXXX and XX/XX/XXXX masks
    text = re.sub(r"\$", " dollar_amount ", text)   # leftover $ from masked amounts
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()                      # collapse extra spaces
    return text

df["text"] = df["narrative"].apply(clean_text)
df["n_words"] = df["text"].str.split().str.len()

# drop near-empty complaints (fewer than 5 words)
before = len(df)
df = df[df["n_words"] >= 5].reset_index(drop=True)
print(f"Dropped {before - len(df)} very short complaints")

print(df["text"].iloc[0][:400])

Dropped 0 very short complaints
i was booking a flight on the website on for a flight trip on it showed me a promotional where it showed if i were to apply for the " delta skymiles gold american express card '' and get approved i would dollar_amount statement credit fir this flight booking and would be able to use the card this booking. but aftwr i got approved i was not given any instant account information and after spending 2


In [23]:
MERGE_FROM = "Incorrect information on your report"
MERGE_TO   = "Credit reporting"

df["label"] = df["label"].replace(MERGE_FROM, MERGE_TO)

# rebuild the splits so train/valid/test pick up the new label
train = df[df["split"] == "train"]
valid = df[df["split"] == "valid"]
test  = df[df["split"] == "test"]
y_train, y_valid, y_test = train["label"], valid["label"], test["label"]

print(df["label"].nunique(), "classes")          # expect 10
print(df["label"].value_counts())
print(len(train), len(valid), len(test))

10 classes
label
Problem with a purchase shown on your statement            22264
Other features, terms, or problems                         10956
Fees or interest                                            9780
Getting a credit card                                       8574
Problem when making payments                                8143
Closing your account                                        6579
Advertising and marketing, including promotional offers     5323
Credit reporting                                            4925
Trouble using your card                                     3933
Struggling to pay your bill                                 1469
Name: count, dtype: int64
45102 14652 22192


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(
    ngram_range=(1, 2),   # single words + word pairs ("late fee", "not received")
    min_df=5,             # ignore terms in fewer than 5 complaints
    max_df=0.9,           # ignore terms in more than 90% of complaints
    sublinear_tf=True     # 1 + log(tf): repeated words count less
)

X_train = vec.fit_transform(train["text"])   # learn vocabulary + IDF from TRAIN only
X_valid = vec.transform(valid["text"])       # apply the same vocabulary
X_test  = vec.transform(test["text"])

print("Train matrix:", X_train.shape)
print("Vocabulary size:", len(vec.vocabulary_))

Train matrix: (45102, 168964)
Vocabulary size: 168964


In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

lr = LogisticRegression(max_iter=1000, C = 3.0,random_state = 42)
lr.fit(X_train, y_train)            # may take a few minutes

pred_lr = lr.predict(X_valid)
print("Logistic regression  macro-F1:", round(f1_score(y_valid, pred_lr, average="macro"), 3))
print(classification_report(y_valid, pred_lr, digits=3))

Logistic regression  macro-F1: 0.614
                                                         precision    recall  f1-score   support

Advertising and marketing, including promotional offers      0.678     0.547     0.605       788
                                   Closing your account      0.655     0.611     0.632      1174
                                       Credit reporting      0.597     0.629     0.613      1021
                                       Fees or interest      0.688     0.738     0.712      1515
                                  Getting a credit card      0.709     0.732     0.720      1583
                     Other features, terms, or problems      0.447     0.469     0.457      1765
                           Problem when making payments      0.635     0.544     0.586      1398
        Problem with a purchase shown on your statement      0.811     0.889     0.848      4396
                            Struggling to pay your bill      0.534     0.400     0.457   

In [22]:
cm = pd.crosstab(y_valid, pred_lr, rownames=["true"], colnames=["predicted"])

two = ["Incorrect information on your report", "Credit reporting"]

# For complaints TRULY in each class: what did the model predict?
print(cm.loc[two].T.sort_values(two[0], ascending=False))

true                                                Incorrect information on your report  \
predicted                                                                                  
Incorrect information on your report                                                 100   
Getting a credit card                                                                 64   
Problem with a purchase shown on your statement                                       51   
Problem when making payments                                                          50   
Other features, terms, or problems                                                    49   
Closing your account                                                                  34   
Fees or interest                                                                      22   
Struggling to pay your bill                                                            7   
Credit reporting                                                                

In [15]:
# Complaints PREDICTED as "Incorrect information": what were they truly?
print(cm["Incorrect information on your report"].sort_values(ascending=False).head(5))

true
Problem when making payments            152
Incorrect information on your report    100
Credit reporting                        100
Getting a credit card                    18
Closing your account                     14
Name: Incorrect information on your report, dtype: int64


In [16]:
yearly = pd.crosstab(df["date_received"].dt.year, df["label"])[two]
print(yearly)

label          Incorrect information on your report  Credit reporting
date_received                                                        
2017                                            186               124
2018                                            302               202
2019                                            284               446
2020                                            391               609
2021                                            382               639
2022                                            559               579
2023                                             51               171


In [17]:
print(lr.n_iter_)   # e.g. [212]: far below 1000

[149]


In [18]:
for C in [0.3, 1, 3, 10, 30]:
    m = LogisticRegression(max_iter=1000, C=C).fit(X_train, y_train)
    p = m.predict(X_valid)
    print(f"C={C:<5} valid macro-F1: {f1_score(y_valid, p, average='macro'):.3f}")

C=0.3   valid macro-F1: 0.529
C=1     valid macro-F1: 0.574
C=3     valid macro-F1: 0.589
C=10    valid macro-F1: 0.584


KeyboardInterrupt: 

In [27]:
cw = "balanced"   # or None, whichever won in step 1
for C in [2,4]:
    m = LogisticRegression(max_iter=1000, C=C, random_state=42).fit(X_train, y_train)
    p = m.predict(X_valid)
    print(f"C={C:<3} macro-F1: {f1_score(y_valid, p, average='macro'):.3f}")

C=2   macro-F1: 0.613
C=4   macro-F1: 0.614


In [28]:
cw = "balanced"   # or None, whichever won in step 1
for C in [1,2,3,5]:
    m = LogisticRegression(max_iter=1000, C=C,class_weight = cw, random_state=42).fit(X_train, y_train)
    p = m.predict(X_valid)
    print(f"C={C:<3} macro-F1: {f1_score(y_valid, p, average='macro'):.3f}")

C=1   macro-F1: 0.633
C=2   macro-F1: 0.635
C=3   macro-F1: 0.631
C=5   macro-F1: 0.629


In [29]:
lr_final = LogisticRegression(max_iter=1000, C=2, class_weight="balanced", random_state=42)
lr_final.fit(X_train, y_train)
pred_final = lr_final.predict(X_valid)

print("Final LR  macro-F1:", round(f1_score(y_valid, pred_final, average="macro"), 3),
      " accuracy:", round((pred_final == y_valid).mean(), 3))
print(classification_report(y_valid, pred_final, digits=3))

Final LR  macro-F1: 0.635  accuracy: 0.69
                                                         precision    recall  f1-score   support

Advertising and marketing, including promotional offers      0.631     0.636     0.633       788
                                   Closing your account      0.627     0.653     0.640      1174
                                       Credit reporting      0.560     0.703     0.623      1021
                                       Fees or interest      0.706     0.704     0.705      1515
                                  Getting a credit card      0.694     0.775     0.733      1583
                     Other features, terms, or problems      0.505     0.422     0.460      1765
                           Problem when making payments      0.656     0.554     0.601      1398
        Problem with a purchase shown on your statement      0.884     0.839     0.861      4396
                            Struggling to pay your bill      0.440     0.663     0.5

In [30]:
from sklearn.naive_bayes import MultinomialNB

for fit_prior in [True, False]:
    nb = MultinomialNB(alpha=1.0, fit_prior=fit_prior)
    nb.fit(X_train, y_train)
    p = nb.predict(X_valid)
    print(f"Naive Bayes fit_prior={fit_prior}  macro-F1: {f1_score(y_valid, p, average='macro'):.3f}  "
          f"accuracy: {(p == y_valid).mean():.3f}")

Naive Bayes fit_prior=True  macro-F1: 0.232  accuracy: 0.421
Naive Bayes fit_prior=False  macro-F1: 0.302  accuracy: 0.472


In [31]:
from sklearn.naive_bayes import MultinomialNB, ComplementNB

for alpha in [1.0,0.3, 0.1, 0.03, 0.01]:
    for name, Model in [("Multinomial", MultinomialNB), ("Complement", ComplementNB)]:
        nb = Model(alpha=alpha) if name == "Complement" else Model(alpha=alpha, fit_prior=False)
        nb.fit(X_train, y_train)
        p = nb.predict(X_valid)
        print(f"{name:<11} alpha={alpha:<5} macro-F1: {f1_score(y_valid, p, average='macro'):.3f}  "
              f"accuracy: {(p == y_valid).mean():.3f}")

Multinomial alpha=1.0   macro-F1: 0.302  accuracy: 0.472
Complement  alpha=1.0   macro-F1: 0.529  accuracy: 0.626
Multinomial alpha=0.3   macro-F1: 0.465  accuracy: 0.603
Complement  alpha=0.3   macro-F1: 0.562  accuracy: 0.649
Multinomial alpha=0.1   macro-F1: 0.556  accuracy: 0.642
Complement  alpha=0.1   macro-F1: 0.560  accuracy: 0.647
Multinomial alpha=0.03  macro-F1: 0.574  accuracy: 0.644
Complement  alpha=0.03  macro-F1: 0.555  accuracy: 0.642
Multinomial alpha=0.01  macro-F1: 0.563  accuracy: 0.633
Complement  alpha=0.01  macro-F1: 0.554  accuracy: 0.641


In [34]:
from sklearn.naive_bayes import MultinomialNB, ComplementNB

for alpha in [0.08,0.07,0.06,0.05]:
    for name, Model in [("Multinomial", MultinomialNB), ("Complement", ComplementNB)]:
        nb = Model(alpha=alpha) if name == "Complement" else Model(alpha=alpha, fit_prior=False)
        nb.fit(X_train, y_train)
        p = nb.predict(X_valid)
        print(f"{name:<11} alpha={alpha:<5} macro-F1: {f1_score(y_valid, p, average='macro'):.3f}  "
              f"accuracy: {(p == y_valid).mean():.3f}")

Multinomial alpha=0.08  macro-F1: 0.570  accuracy: 0.646
Complement  alpha=0.08  macro-F1: 0.560  accuracy: 0.646
Multinomial alpha=0.07  macro-F1: 0.572  accuracy: 0.646
Complement  alpha=0.07  macro-F1: 0.560  accuracy: 0.646
Multinomial alpha=0.06  macro-F1: 0.576  accuracy: 0.647
Complement  alpha=0.06  macro-F1: 0.559  accuracy: 0.645
Multinomial alpha=0.05  macro-F1: 0.577  accuracy: 0.647
Complement  alpha=0.05  macro-F1: 0.558  accuracy: 0.645


In [35]:
# final choice Multinomial,aplha = 0.05,fit_prior = False provides macro F1 = 0.577 and accuracy = 0.647